# CardioAI Assignment Notebook — Heart Disease Mining + MNIST CNN + Dashboard

This notebook completes the full assignment:

- Preprocessing and data setup for the UCI Cleveland Heart Disease dataset
- Part A: K-Means, hierarchical clustering, PCA, t-SNE
- Part B: Random Forest, XGBoost, SHAP, ROC comparison
- Part C: SLP, MLP, 5-fold CV, ablation study
- Part D: CNN on MNIST subset
- Part E: Streamlit dashboard file generation

Upload `processed.cleveland.data` in Colab when prompted. The notebook also creates `app.py`, model files, `requirements.txt`, and a ZIP for dashboard submission.

In [ ]:
!pip -q install xgboost shap imbalanced-learn streamlit joblib

import os, time, random, json, warnings, zipfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import *
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb
import shap
import joblib

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks, optimizers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Required Preprocessing

In [ ]:
DATA_PATH = "processed.cleveland.data"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Upload processed.cleveland.data")
        files.upload()
    except Exception:
        print("Place processed.cleveland.data in the notebook folder.")

columns = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
           "exang","oldpeak","slope","ca","thal","target"]

df_raw = pd.read_csv(DATA_PATH, header=None, names=columns, na_values="?")
print("Original shape:", df_raw.shape)
display(df_raw.head())
print("\nData types:")
print(df_raw.dtypes)

missing = df_raw.isna().sum()
print("\nMissing values by affected column:")
print(missing[missing > 0] if (missing > 0).any() else "No missing values")

print("Rows with any missing value:", df_raw.isna().any(axis=1).sum())

df = df_raw.dropna().copy()
for c in columns:
    df[c] = pd.to_numeric(df[c])
df["target"] = (df["target"] > 0).astype(int)

print("Final retained row count:", len(df))
print("Final shape:", df.shape)
display(df.head())

In [ ]:
class_dist = df["target"].value_counts().sort_index().to_frame("count")
class_dist["percentage"] = 100 * class_dist["count"] / len(df)
display(class_dist)

minority_ratio = class_dist["count"].min() / class_dist["count"].max()
USE_SMOTE = minority_ratio < 0.80
print(f"Minority/Majority ratio: {minority_ratio:.3f}")
print("SMOTE selected:", USE_SMOTE)
print("Justification: If the minority/majority ratio is near or above 0.80, the dataset is reasonably balanced; otherwise SMOTE is applied only to training folds.")

In [ ]:
categorical_cols = ["cp", "restecg", "slope", "thal"]
continuous_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
binary_cols = ["sex", "fbs", "exang"]
feature_cols = categorical_cols + continuous_cols + binary_cols

X = df[feature_cols].copy()
y = df["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

try:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")

preprocessor = ColumnTransformer([
    ("cat", ohe, categorical_cols),
    ("num", StandardScaler(), continuous_cols),
    ("bin", "passthrough", binary_cols)
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)
X_all_prep = preprocessor.transform(X)

feature_names = preprocessor.get_feature_names_out()
print("Train:", X_train_prep.shape, "Test:", X_test_prep.shape)
print("Features:", list(feature_names))

In [ ]:
numeric_original_cols = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach","exang","oldpeak","slope","ca","thal"]
corr = df[numeric_original_cols].corr()

plt.figure(figsize=(12, 9))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap of Original Numeric Features")
plt.tight_layout()
plt.show()

pairs = []
for i, a in enumerate(numeric_original_cols):
    for b in numeric_original_cols[i+1:]:
        pairs.append([a, b, corr.loc[a,b], abs(corr.loc[a,b])])
top3_corr = pd.DataFrame(pairs, columns=["feature_1","feature_2","correlation","abs_correlation"]).sort_values("abs_correlation", ascending=False).head(3)
display(top3_corr)
print("Naive Bayes note: strong correlations violate the conditional independence assumption and may cause double-counting of related evidence.")

# Part A — Unsupervised Learning

In [ ]:
ks = list(range(2, 9))
wcss, sils = [], []
models_k = {}

for k in ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    labels = km.fit_predict(X_all_prep)
    models_k[k] = (km, labels)
    wcss.append(km.inertia_)
    sils.append(silhouette_score(X_all_prep, labels))

chosen_k = ks[int(np.argmax(sils))]
chosen_km, kmeans_labels = models_k[chosen_k]

fig, ax1 = plt.subplots(figsize=(10,6))
ax1.plot(ks, wcss, marker="o", label="WCSS/Inertia")
ax1.set_xlabel("k")
ax1.set_ylabel("WCSS")
ax1.axvline(chosen_k, linestyle="--", label=f"Chosen k={chosen_k}")

ax2 = ax1.twinx()
ax2.plot(ks, sils, marker="s", label="Silhouette")
ax2.set_ylabel("Silhouette Score")
h1,l1=ax1.get_legend_handles_labels()
h2,l2=ax2.get_legend_handles_labels()
ax1.legend(h1+h2,l1+l2)
plt.title("K-Means WCSS and Silhouette Curves")
plt.tight_layout()
plt.show()

print(f"Chosen k = {chosen_k}. This value maximises silhouette score among tested k values, giving the best compactness/separation tradeoff while remaining interpretable.")

In [ ]:
pca2 = PCA(n_components=2, random_state=SEED)
X_pca2 = pca2.fit_transform(X_all_prep)

fig, axes = plt.subplots(1,2,figsize=(14,5))
s1 = axes[0].scatter(X_pca2[:,0], X_pca2[:,1], c=kmeans_labels, s=35)
axes[0].set_title("PCA colored by K-Means cluster")
plt.colorbar(s1, ax=axes[0])
s2 = axes[1].scatter(X_pca2[:,0], X_pca2[:,1], c=y, s=35)
axes[1].set_title("PCA colored by true disease label")
plt.colorbar(s2, ax=axes[1])
plt.tight_layout()
plt.show()

ari_k_true = adjusted_rand_score(y, kmeans_labels)
print("K-Means vs true label ARI:", round(ari_k_true, 4))
print("Interpretation: ARI near 0 means weak natural alignment with clinical labels; higher positive ARI means better agreement beyond chance.")

In [ ]:
cluster_df = df.copy()
cluster_df["cluster"] = kmeans_labels
summary = cluster_df.groupby("cluster").agg(
    cluster_size=("target","size"),
    disease_proportion=("target","mean"),
    mean_thalach=("thalach","mean"),
    mean_oldpeak=("oldpeak","mean"),
    mean_cp=("cp","mean")
).reset_index()
display(summary)

for _, r in summary.iterrows():
    profile = "higher-risk" if r["disease_proportion"] >= summary["disease_proportion"].median() else "lower-risk"
    print(f"Cluster {int(r['cluster'])}: {profile}; disease proportion={r['disease_proportion']:.2f}, thalach={r['mean_thalach']:.1f}, oldpeak={r['mean_oldpeak']:.2f}, cp={r['mean_cp']:.2f}.")

In [ ]:
Z = linkage(X_all_prep, method="ward")
hier_k = chosen_k
hier_labels = fcluster(Z, t=hier_k, criterion="maxclust") - 1
cut_height = Z[-(hier_k-1), 2] if hier_k > 1 else Z[-1,2]

plt.figure(figsize=(14,6))
dendrogram(Z, truncate_mode="lastp", p=25, leaf_rotation=45, show_contracted=True)
plt.axhline(cut_height, linestyle="--", label=f"Cut height ≈ {cut_height:.2f}; clusters={hier_k}")
plt.title("Ward Linkage Dendrogram, Top 25 Merges")
plt.xlabel("Merged groups")
plt.ylabel("Ward distance")
plt.legend()
plt.tight_layout()
plt.show()

ct = pd.crosstab(pd.Series(hier_labels, name="Hier Cluster"), pd.Series(y, name="True Label"))
display(ct)

ari_k_h = adjusted_rand_score(kmeans_labels, hier_labels)
print("ARI between K-Means and Hierarchical clusters:", round(ari_k_h, 4))
print("Clinical segmentation discussion: K-Means is easier to operationalise and compare across runs. Hierarchical clustering is useful for seeing nested patient similarity. I would trust the method with clearer, more stable cluster profiles and stronger interpretability.")

In [ ]:
pca_full = PCA(random_state=SEED)
X_pca_full = pca_full.fit_transform(X_all_prep)
explained = pca_full.explained_variance_ratio_
cum = np.cumsum(explained)
n90 = np.argmax(cum >= 0.90) + 1

plt.figure(figsize=(10,6))
plt.plot(range(1,len(explained)+1), explained, marker="o", label="Individual")
plt.plot(range(1,len(cum)+1), cum, marker="s", label="Cumulative")
plt.axhline(0.90, linestyle="--")
plt.axvline(n90, linestyle="--", label=f"90% at {n90} components")
plt.xlabel("Component")
plt.ylabel("Explained variance ratio")
plt.title("PCA Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()
print("Components for 90% explained variance:", n90)

X_tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca", learning_rate="auto").fit_transform(X_all_prep)
plt.figure(figsize=(8,6))
s=plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, s=35)
plt.colorbar(s, label="Disease label")
plt.title("t-SNE colored by disease label")
plt.tight_layout()
plt.show()
print("If colors overlap, the two classes are not cleanly separable geometrically, which means supervised classification must learn subtle patterns.")

# Part B — Bagging & Boosting

In [ ]:
def maybe_smote_model(est):
    return ImbPipeline([("smote", SMOTE(random_state=SEED)), ("model", est)]) if USE_SMOTE else est

def evaluate_classifier(name, model, X_te, y_te, plot_cm=True):
    y_pred = model.predict(X_te)
    y_score = model.predict_proba(X_te)[:,1]
    out = {
        "Classifier": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Macro Precision": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_te, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "AUC-ROC": roc_auc_score(y_te, y_score),
        "Recall (Disease)": recall_score(y_te, y_pred, pos_label=1, zero_division=0)
    }
    print("\n", name)
    print(pd.Series(out))
    print(classification_report(y_te, y_pred, target_names=["No Disease","Disease Present"], zero_division=0))
    cm = confusion_matrix(y_te, y_pred)
    if plot_cm:
        ConfusionMatrixDisplay(cm, display_labels=["No Disease","Disease Present"]).plot(values_format="d")
        plt.title(f"Confusion Matrix — {name}")
        plt.show()
    return out, cm, y_score

start=time.time()
base = maybe_smote_model(LogisticRegression(max_iter=2000, random_state=SEED))
base.fit(X_train_prep, y_train)
base_time=time.time()-start
base_metrics, base_cm, base_scores = evaluate_classifier("Logistic Regression Baseline", base, X_test_prep, y_test)
base_metrics["Train Time"] = base_time

In [ ]:
rf = RandomForestClassifier(random_state=SEED, oob_score=True, bootstrap=True)
rf_pipe = maybe_smote_model(rf)
if USE_SMOTE:
    grid_rf = {"model__n_estimators":[50,100,200], "model__max_depth":[None,5,10]}
else:
    grid_rf = {"n_estimators":[50,100,200], "max_depth":[None,5,10]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
gs_rf = GridSearchCV(rf_pipe, grid_rf, scoring="f1_macro", cv=cv, n_jobs=-1)
start=time.time()
gs_rf.fit(X_train_prep, y_train)
rf_time=time.time()-start
best_rf=gs_rf.best_estimator_
print("Best RF params:", gs_rf.best_params_)
print("Best RF CV macro F1:", round(gs_rf.best_score_,4))
rf_metrics, rf_cm, rf_scores = evaluate_classifier("Random Forest", best_rf, X_test_prep, y_test)
rf_metrics["Train Time"] = rf_time

In [ ]:
oob_errors=[]
for n in range(1,201):
    clf=RandomForestClassifier(n_estimators=n, random_state=SEED, oob_score=True, bootstrap=True, n_jobs=-1)
    clf.fit(X_train_prep, y_train)
    oob_errors.append(1-clf.oob_score_)
stabilise_at=200
for i in range(20,len(oob_errors)):
    if abs(np.mean(oob_errors[i-10:i])-np.mean(oob_errors[i-20:i-10])) < 0.002:
        stabilise_at=i+1
        break
plt.figure(figsize=(10,6))
plt.plot(range(1,201), oob_errors)
plt.axvline(stabilise_at, linestyle="--", label=f"Stabilises around {stabilise_at}")
plt.xlabel("Trees")
plt.ylabel("OOB error")
plt.title("Random Forest OOB Error")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
rf_imp_model = best_rf.named_steps["model"] if USE_SMOTE else best_rf
imp = pd.DataFrame({"feature":feature_names, "importance":rf_imp_model.feature_importances_}).sort_values("importance")
plt.figure(figsize=(10,8))
plt.barh(imp["feature"], imp["importance"])
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

top5_rf = imp.sort_values("importance", ascending=False).head(5)
display(top5_rf)
for f in top5_rf["feature"]:
    print(f"- {f}: clinically plausible because it represents symptoms, cardiac stress response, vessel involvement, ECG/ST change, or known risk structure.")
print("Recall consequence: false negatives for disease present are dangerous because a sick patient may be told they are low-risk and miss further cardiac evaluation.")

In [ ]:
xgb_base = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=SEED,
                             n_estimators=300, subsample=0.9, colsample_bytree=0.9, tree_method="hist")
grid_xgb = {"learning_rate":[0.01,0.1,0.3], "max_depth":[3,5,7]}
gs_xgb = GridSearchCV(xgb_base, grid_xgb, scoring="f1_macro", cv=cv, n_jobs=-1)
start=time.time()
gs_xgb.fit(X_train_prep, y_train)
xgb_cv_time=time.time()-start
print("Model choice: XGBoost")
print("Best XGBoost params:", gs_xgb.best_params_)
print("Best XGBoost CV macro F1:", round(gs_xgb.best_score_,4))

X_tr2, X_val2, y_tr2, y_val2 = train_test_split(X_train_prep, y_train, test_size=0.2, stratify=y_train, random_state=SEED)
best_params = gs_xgb.best_params_
xgb_final = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=SEED,
                              n_estimators=500, learning_rate=best_params["learning_rate"],
                              max_depth=best_params["max_depth"], subsample=0.9, colsample_bytree=0.9,
                              tree_method="hist", early_stopping_rounds=50)
start=time.time()
xgb_final.fit(X_tr2, y_tr2, eval_set=[(X_tr2,y_tr2),(X_val2,y_val2)], verbose=False)
xgb_time=time.time()-start+xgb_cv_time

res=xgb_final.evals_result()
train_loss=res["validation_0"]["logloss"]
val_loss=res["validation_1"]["logloss"]
best_round=getattr(xgb_final,"best_iteration",int(np.argmin(val_loss)))
plt.figure(figsize=(10,6))
plt.plot(train_loss,label="train")
plt.plot(val_loss,label="validation")
plt.axvline(best_round, linestyle="--", label=f"Best round {best_round}")
plt.xlabel("Boosting round")
plt.ylabel("Log-loss")
plt.title("XGBoost Log-loss")
plt.legend()
plt.tight_layout()
plt.show()
print("Overfitting note: if train loss decreases while validation loss rises, the model is overfitting; early stopping prevents continuing beyond the best validation round.")

xgb_metrics, xgb_cm, xgb_scores = evaluate_classifier("XGBoost", xgb_final, X_test_prep, y_test)
xgb_metrics["Train Time"] = xgb_time

In [ ]:
explainer = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X_test_prep)
shap.summary_plot(shap_values, X_test_prep, feature_names=feature_names, show=False)
plt.title("XGBoost SHAP Summary")
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame([base_metrics, rf_metrics, xgb_metrics])
comparison = comparison[["Classifier","Accuracy","Macro F1","AUC-ROC","Recall (Disease)","Train Time"]]
display(comparison)

plt.figure(figsize=(9,7))
for name, scores in [("Logistic Regression Baseline", base_scores), ("Random Forest", rf_scores), ("XGBoost", xgb_scores)]:
    fpr,tpr,_=roc_curve(y_test, scores)
    auc=roc_auc_score(y_test, scores)
    plt.plot(fpr,tpr,label=f"{name} AUC={auc:.3f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()

rec = comparison.sort_values(["Recall (Disease)","AUC-ROC","Macro F1"], ascending=False).iloc[0]["Classifier"]
print("Recommended model:", rec)
print("Recommendation: In screening, recall for disease matters more than overall accuracy because missed disease cases can delay treatment. I would deploy the model with the best disease recall and strong AUC, while still requiring interpretability. RF/XGBoost are attractive because feature importance and SHAP explanations can support clinical review. The model should be used as a screening support tool, not as a final diagnosis.")

# Part C — Artificial Neural Networks on Tabular Data

In [ ]:
def keras_metrics(y_true, probs, threshold=0.5):
    pred=(probs>=threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true,pred),
        "F1": f1_score(y_true,pred,zero_division=0),
        "Macro F1": f1_score(y_true,pred,average="macro",zero_division=0),
        "AUC": roc_auc_score(y_true,probs)
    }, confusion_matrix(y_true,pred)

def plot_history(hist,title):
    plt.figure(figsize=(10,5))
    plt.plot(hist.history["loss"],label="train loss")
    if "val_loss" in hist.history: plt.plot(hist.history["val_loss"],label="val loss")
    plt.title(title+" Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout(); plt.show()
    plt.figure(figsize=(10,5))
    plt.plot(hist.history["accuracy"],label="train acc")
    if "val_accuracy" in hist.history: plt.plot(hist.history["val_accuracy"],label="val acc")
    plt.title(title+" Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.tight_layout(); plt.show()

input_dim=X_train_prep.shape[1]
slp=models.Sequential([layers.Input(shape=(input_dim,)), layers.Dense(1,activation="sigmoid")])
slp.compile(optimizer=optimizers.SGD(learning_rate=0.01), loss="binary_crossentropy", metrics=["accuracy"])
hist_slp=slp.fit(X_train_prep,y_train,epochs=100,batch_size=16,verbose=0)
plot_history(hist_slp,"SLP")

w=slp.layers[0].get_weights()[0].reshape(-1)
wdf=pd.DataFrame({"feature":feature_names,"weight":w,"abs_weight":np.abs(w)}).sort_values("abs_weight",ascending=False)
display(wdf.head(10))
print("Top 3 SLP weights:", wdf.head(3)["feature"].tolist())
print("RF top 5:", top5_rf["feature"].tolist())

slp_probs=slp.predict(X_test_prep,verbose=0).ravel()
slp_metrics, slp_cm=keras_metrics(y_test,slp_probs)
print(slp_metrics)
ConfusionMatrixDisplay(slp_cm, display_labels=["No Disease","Disease Present"]).plot(values_format="d")
plt.title("SLP Confusion Matrix")
plt.show()
print("A linear model is limited because it can only draw one linear boundary and cannot capture complex nonlinear interactions among symptoms, ECG signals, and stress-test variables.")

In [ ]:
def build_mlp(arch="Small", activation="relu", dropout=0.25, l2=1e-4, lr=0.001):
    units={"Small":[32],"Medium":[64,32],"Large":[128,64,32],"Custom":[64,64,16]}[arch]
    m=models.Sequential([layers.Input(shape=(input_dim,))])
    for u in units:
        m.add(layers.Dense(u, activation=activation, kernel_regularizer=regularizers.l2(l2)))
        if dropout>0: m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1,activation="sigmoid"))
    m.compile(optimizer=optimizers.Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    return m

X_tr_nn,X_val_nn,y_tr_nn,y_val_nn=train_test_split(X_train_prep,y_train,test_size=0.2,stratify=y_train,random_state=SEED)
arch_rows=[]; histories={}; models_arch={}
for arch in ["Small","Medium","Large","Custom"]:
    tf.random.set_seed(SEED)
    m=build_mlp(arch)
    es=callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    start=time.time()
    h=m.fit(X_tr_nn,y_tr_nn,validation_data=(X_val_nn,y_val_nn),epochs=150,batch_size=16,verbose=0,callbacks=[es])
    t=time.time()-start
    probs=m.predict(X_val_nn,verbose=0).ravel()
    f1=f1_score(y_val_nn,(probs>=0.5).astype(int),zero_division=0)
    arch_rows.append({"Architecture":arch,"Val F1":f1,"Training Time":t,"Epochs Run":len(h.history["loss"])})
    histories[arch]=h; models_arch[arch]=m
arch_df=pd.DataFrame(arch_rows)
display(arch_df)
best_arch=arch_df.sort_values("Val F1",ascending=False).iloc[0]["Architecture"]
print("Best architecture:", best_arch)

In [ ]:
final_mlp=models_arch[best_arch]
final_hist=histories[best_arch]
plot_history(final_hist, f"Final MLP {best_arch}")
print("Design: ReLU activation, Dropout=0.25, L2=1e-4, Adam lr=0.001, EarlyStopping patience=10.")
print("Early stopping epoch:", len(final_hist.history["loss"]))

mlp_probs=final_mlp.predict(X_test_prep,verbose=0).ravel()
mlp_metrics, mlp_cm=keras_metrics(y_test,mlp_probs)
print(mlp_metrics)
ConfusionMatrixDisplay(mlp_cm, display_labels=["No Disease","Disease Present"]).plot(values_format="d")
plt.title("Final MLP Confusion Matrix")
plt.show()

cv_acc=[]; cv_f1=[]
skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold,(tr,va) in enumerate(skf.split(X_train_prep,y_train),1):
    tf.random.set_seed(SEED+fold)
    m=build_mlp(best_arch)
    es=callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    m.fit(X_train_prep[tr], y_train.iloc[tr], validation_data=(X_train_prep[va], y_train.iloc[va]), epochs=150, batch_size=16, verbose=0, callbacks=[es])
    pred=(m.predict(X_train_prep[va],verbose=0).ravel()>=0.5).astype(int)
    cv_acc.append(accuracy_score(y_train.iloc[va],pred))
    cv_f1.append(f1_score(y_train.iloc[va],pred,zero_division=0))
print(f"5-fold CV Accuracy: {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"5-fold CV F1: {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")
print("MLP comparison: MLP can model nonlinear combinations, but on small tabular data RF/XGBoost often generalise better and are easier to explain with SHAP or feature importance.")

In [ ]:
def build_variant(variant):
    activation="relu"; dropout=0.25; use_es=True
    if variant=="No Dropout": dropout=0.0
    if variant=="Sigmoid Activations": activation="sigmoid"
    if variant=="No Early Stopping": use_es=False
    return build_mlp(best_arch, activation=activation, dropout=dropout), use_es

abl=[{"Variant":"Best MLP","Test F1":mlp_metrics["F1"],"Test Accuracy":mlp_metrics["Accuracy"],"Epochs":len(final_hist.history["loss"])}]
abl_hist={"Best MLP":final_hist}
for variant in ["No Dropout","Sigmoid Activations","No Early Stopping"]:
    tf.random.set_seed(SEED)
    m,use_es=build_variant(variant)
    cbs=[callbacks.EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)] if use_es else []
    h=m.fit(X_tr_nn,y_tr_nn,validation_data=(X_val_nn,y_val_nn),epochs=150,batch_size=16,verbose=0,callbacks=cbs)
    probs=m.predict(X_test_prep,verbose=0).ravel()
    met,_=keras_metrics(y_test,probs)
    abl.append({"Variant":variant,"Test F1":met["F1"],"Test Accuracy":met["Accuracy"],"Epochs":len(h.history["loss"])})
    abl_hist[variant]=h
abl_df=pd.DataFrame(abl)
display(abl_df)

plt.figure(figsize=(10,6))
for name,h in abl_hist.items():
    if "val_loss" in h.history:
        plt.plot(h.history["val_loss"], label=name)
plt.title("Ablation Validation Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()
print("The component whose removal gives the largest F1 drop contributed most to performance.")

# Part D — CNN on MNIST

In [ ]:
(x_train_m,y_train_m),(x_test_m,y_test_m)=mnist.load_data()
x_train_m,y_train_m=x_train_m[:12000],y_train_m[:12000]
x_test_m,y_test_m=x_test_m[:2000],y_test_m[:2000]

x_train_m=x_train_m.astype("float32")/255.0
x_test_m=x_test_m.astype("float32")/255.0
x_train_cnn=x_train_m[...,None]
x_test_cnn=x_test_m[...,None]
y_train_cat=to_categorical(y_train_m,10)
y_test_cat=to_categorical(y_test_m,10)

fig,axes=plt.subplots(5,2,figsize=(6,10))
for d in range(10):
    idx=np.where(y_train_m==d)[0][0]
    ax=axes[d//2,d%2]
    ax.imshow(x_train_m[idx],cmap="gray")
    ax.set_title(f"Digit {d}")
    ax.axis("off")
plt.tight_layout()
plt.show()

mnist_mlp=models.Sequential([layers.Input(shape=(28,28)),layers.Flatten(),layers.Dense(64,activation="relu"),layers.Dense(10,activation="softmax")])
mnist_mlp.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
hist_mnist_mlp=mnist_mlp.fit(x_train_m,y_train_cat,validation_split=0.1,epochs=5,batch_size=64,verbose=1)
_, mlp_test_acc=mnist_mlp.evaluate(x_test_m,y_test_cat,verbose=0)
print("MNIST MLP baseline test accuracy:", mlp_test_acc)

In [ ]:
data_aug=models.Sequential([layers.RandomRotation(0.08),layers.RandomZoom(0.08),layers.RandomTranslation(0.08,0.08)])

cnn=models.Sequential([
    layers.Input(shape=(28,28,1)),
    data_aug,
    layers.Conv2D(16,(3,3),activation="relu",padding="same",name="conv1"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32,(3,3),activation="relu",padding="same",name="conv2"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64,activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10,activation="softmax")
])
cnn.compile(optimizer=optimizers.Adam(learning_rate=0.001),loss="categorical_crossentropy",metrics=["accuracy"])
cnn.summary()
hist_cnn=cnn.fit(x_train_cnn,y_train_cat,validation_split=0.1,epochs=12,batch_size=64,verbose=1)

plt.figure(figsize=(10,5))
plt.plot(hist_cnn.history["accuracy"],label="train")
plt.plot(hist_cnn.history["val_accuracy"],label="validation")
plt.axhline(mlp_test_acc,linestyle="--",label="MLP baseline")
plt.title("CNN Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,5))
plt.plot(hist_cnn.history["loss"],label="train")
plt.plot(hist_cnn.history["val_loss"],label="validation")
plt.title("CNN Loss")
plt.legend()
plt.tight_layout()
plt.show()

_,cnn_acc=cnn.evaluate(x_test_cnn,y_test_cat,verbose=0)
prob=cnn.predict(x_test_cnn,verbose=0)
pred=np.argmax(prob,axis=1)
cnn_f1=f1_score(y_test_m,pred,average="macro")
print("CNN test accuracy:", cnn_acc)
print("CNN macro F1:", cnn_f1)

cm=confusion_matrix(y_test_m,pred)
plt.figure(figsize=(9,7))
plt.imshow(cm)
plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("CNN Confusion Matrix")
for i in range(10):
    for j in range(10):
        plt.text(j,i,cm[i,j],ha="center",va="center")
plt.tight_layout()
plt.show()

conf=[]
for i in range(10):
    for j in range(10):
        if i!=j: conf.append((i,j,cm[i,j]))
print("Two most confused digit pairs:", sorted(conf,key=lambda x:x[2],reverse=True)[:2])
surpass=None
for e,a in enumerate(hist_cnn.history["val_accuracy"],1):
    if a>=mlp_test_acc:
        surpass=e; break
print("CNN surpasses MLP baseline by epoch:", surpass)
print("Visually, confused digits often share loops, curves, or similar slanted strokes.")

In [ ]:
conv1=cnn.get_layer("conv1")
filters,_=conv1.get_weights()
fig,axes=plt.subplots(4,4,figsize=(8,8))
for i in range(16):
    ax=axes[i//4,i%4]
    ax.imshow(filters[:,:,0,i],cmap="gray")
    ax.set_title(f"Filter {i+1}")
    ax.axis("off")
plt.suptitle("First Conv2D Filters")
plt.tight_layout()
plt.show()
print("Early filters typically detect tiny edges, curves, blobs, and stroke directions.")

feature_model=models.Model(inputs=cnn.input, outputs=conv1.output)
for d in range(10):
    idx=np.where(y_test_m==d)[0][0]
    fmap=feature_model.predict(x_test_cnn[idx:idx+1],verbose=0)[0]
    fig,axes=plt.subplots(2,4,figsize=(10,5))
    for ch in range(8):
        ax=axes[ch//4,ch%4]
        ax.imshow(fmap[:,:,ch],cmap="gray")
        ax.set_title(f"Ch {ch+1}")
        ax.axis("off")
    plt.suptitle(f"Feature Maps for Digit {d}")
    plt.tight_layout()
    plt.show()
print("These maps build trust because they show the CNN responds to meaningful local strokes. Unlike fully connected networks, CNNs preserve spatial neighbourhoods and reuse learned filters across the image.")

# Part E — Generate Local Streamlit Dashboard

In [ ]:
final_dashboard_model = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="logloss", random_state=SEED,
    n_estimators=max(50, int(best_round)+1), learning_rate=best_params["learning_rate"],
    max_depth=best_params["max_depth"], subsample=0.9, colsample_bytree=0.9, tree_method="hist"
)
final_dashboard_model.fit(X_train_prep, y_train)

joblib.dump(final_dashboard_model, "heart_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(feature_names, "feature_names.pkl")

sample_patient = X_test.iloc[0].to_dict()
with open("sample_patient.json", "w") as f:
    json.dump(sample_patient, f, indent=2)

requirements = "\n".join(["pandas","numpy","scikit-learn","xgboost","shap","streamlit","joblib","matplotlib"]) + "\n"
with open("requirements.txt", "w") as f:
    f.write(requirements)

app_code = """
import json
import numpy as np
import pandas as pd
import streamlit as st
import joblib
import shap

st.set_page_config(page_title='CardioAI Heart Disease Predictor', layout='wide')
st.title('CardioAI Heart Disease Risk Predictor')
st.write('Educational dashboard for UCI Cleveland Heart Disease screening support.')

model = joblib.load('heart_model.pkl')
preprocessor = joblib.load('preprocessor.pkl')
feature_names = joblib.load('feature_names.pkl')
with open('sample_patient.json') as f:
    sample = json.load(f)

cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal']
hints = {
    'age':'20-80', 'sex':'1 male, 0 female', 'cp':'1-4 chest pain type',
    'trestbps':'90-200 mmHg', 'chol':'100-600 mg/dl', 'fbs':'1 if >120 mg/dl else 0',
    'restecg':'0-2', 'thalach':'70-210', 'exang':'1 yes, 0 no',
    'oldpeak':'0-7', 'slope':'1-3', 'ca':'0-3', 'thal':'3 normal, 6 fixed, 7 reversible'
}
ranges = {
    'age':(20.0,90.0),'sex':(0.0,1.0),'cp':(1.0,4.0),'trestbps':(80.0,220.0),
    'chol':(100.0,650.0),'fbs':(0.0,1.0),'restecg':(0.0,2.0),'thalach':(60.0,230.0),
    'exang':(0.0,1.0),'oldpeak':(0.0,8.0),'slope':(1.0,3.0),'ca':(0.0,3.0),'thal':(3.0,7.0)
}

st.subheader('Input Form')
left, right = st.columns(2)
values = {}
for i, c in enumerate(cols):
    mn, mx = ranges[c]
    box = left if i < 7 else right
    with box:
        values[c] = st.number_input(f'{c} ({hints[c]})', min_value=mn, max_value=mx, value=float(sample[c]), step=0.1 if c=='oldpeak' else 1.0)

if st.button('Predict'):
    X = pd.DataFrame([values])
    Xp = preprocessor.transform(X)
    prob = float(model.predict_proba(Xp)[0,1])
    pred = int(prob >= 0.5)
    label = 'Disease Present' if pred else 'No Disease'
    color = 'red' if pred else 'green'

    st.subheader('Results Panel')
    st.markdown(f\"### Predicted Class: <span style='color:{color}'>{label}</span>\", unsafe_allow_html=True)
    st.metric('Confidence for Disease Present', f'{prob*100:.2f}%')

    explainer = shap.TreeExplainer(model)
    vals = explainer.shap_values(Xp)[0]
    idx = np.argsort(np.abs(vals))[-3:][::-1]
    top = pd.DataFrame({'feature': feature_names[idx], 'impact': vals[idx]})
    st.write('Top 3 features driving the prediction')
    st.bar_chart(top.set_index('feature'))

    if pred:
        st.write('Plain-English explanation: This patient has an elevated screening risk based on the strongest model signals above. A nurse should recommend further cardiac review rather than treating this as a final diagnosis.')
    else:
        st.write('Plain-English explanation: The model estimates lower cardiac risk for this patient profile. Clinical symptoms or abnormal examination findings should still be reviewed by a medical professional.')
"""
with open("app.py", "w") as f:
    f.write(app_code)

with zipfile.ZipFile("cardioai_submission_files.zip", "w") as z:
    for fname in ["app.py","heart_model.pkl","preprocessor.pkl","feature_names.pkl","sample_patient.json","requirements.txt"]:
        z.write(fname)

print("Created app.py, model files, requirements.txt, and cardioai_submission_files.zip")
print("Run locally with: streamlit run app.py")

In [ ]:
try:
    from google.colab import files
    files.download("cardioai_submission_files.zip")
except Exception:
    print("Download cardioai_submission_files.zip manually if not in Colab.")

## Final submission reminder

For the dashboard screenshots, run:

```bash
pip install -r requirements.txt
streamlit run app.py
```

Then include screenshots of:

1. The input form.
2. The prediction result panel.

Medical disclaimer: this is an educational model-building assignment, not a diagnostic medical device.